# Agentic RAG (LangGraph) — service test

Tests the full `AgenticRag` graph: guardrail → retrieve → grade → (rewrite|generate).

**Prereqs:** `docker compose up -d opensearch` with a populated index + `OPENAI_API_KEY`. 
Set `LANGCHAIN_TRACING_V2=true` + `LANGCHAIN_API_KEY` to see traces in LangSmith. 
**Cost:** ~$0.01-0.03 per query (3-5 LLM calls).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.services.agents.factory import make_agentic_rag

agent = make_agentic_rag()   # builds + compiles the graph once
print('graph compiled')

In [ ]:
# Visualize the graph structure (paste into mermaid.live or GitHub markdown)
print(agent.visualize(format='mermaid'))

In [ ]:
# Path 1: OFF-TOPIC query → guardrail should refuse (no retrieval, no answer LLM)
result = await agent.ask('what is the capital of France?')
print('ANSWER:', result['answer'][:200])
print('STEPS :', result['reasoning_steps'])
print('ATTEMPTS:', result['retrieval_attempts'], '(expect 0)')

In [ ]:
# Path 2: CLEAN on-topic query → happy path (guardrail → retrieve → grade yes → generate)
result = await agent.ask('what is multi-head attention in transformers?')
print('ANSWER:', result['answer'][:400])
print()
print('STEPS :', result['reasoning_steps'])
print('ATTEMPTS:', result['retrieval_attempts'], '(expect 1)')

In [ ]:
# Path 3: VAGUE query → may trigger grade-no → rewrite → retry loop
result = await agent.ask('tell me about that new scaling thing')
print('ANSWER:', result['answer'][:300])
print()
print('STEPS :', result['reasoning_steps'])
print('ATTEMPTS:', result['retrieval_attempts'], '(2 = one rewrite loop happened)')